# ACE (1O86) Redocking: Smina and LeDock

Every redocking attempt for ACE so far in this project used AutoDock Vina
(standard and AutoDock4Zn). This notebook tests two genuinely different
docking engines on the exact same system -- not just different Vina
parameters, but different underlying search algorithms and (for Smina)
scoring functions:

- **Smina**: a Vina fork with an extended, customisable scoring function
  and different internal optimisation defaults.
- **LeDock**: an entirely independent docking program (not Vina-derived),
  reported to achieve >90% pose-prediction accuracy on the Astex
  diversity set.

**Purpose:** if the redocking failure persists across three independently
implemented engines (Vina, Smina, LeDock), that is much stronger evidence
of a genuine, engine-independent property of this target than any single
engine's result -- consistent with, and a further test of, the
large-binding-groove explanation already documented (Caballero 2020) for
this target. If either engine succeeds where Vina failed, that is
itself an important, different finding worth investigating further.

Adapted directly from the Jupyter-Dock "Molecular Docking" template's
Smina (Section 5) and LeDock (Section 6) pipelines, using this project's
already-prepared ACE receptor and native ligand files rather than
re-fetching from scratch, for full consistency with every other
redocking result in this study.

**No results are asserted anywhere in this notebook. Every conclusion is
left for after real output is obtained.**

In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
from pymol import cmd
from rdkit import Chem
from rdkit.Chem import rdFMCS

BIN = "/home/ali/medical plants/docking/Jupyter_Dock-main/bin/"

import sys
sys.path.insert(1, 'utilities/')
from utils import getbox, generate_ledock_file, dok_to_sdf

# Change into the same working directory used by every other ACE notebook
# in this project, where the receptor/ligand files already exist. This
# notebook did not do this originally, which caused "Bad input file"
# errors -- RDKit/OS could not find the files because they were never in
# this notebook's default working directory to begin with.
DOCKING_DIR = "/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/"
os.chdir(DOCKING_DIR)
print("Working directory:", os.getcwd())

RECEPTOR_PDB = "1O86_clean_H.pdb"
RECEPTOR_PDBQT = "1O86_clean_H.pdbqt"
NATIVE_LIG_MOL2 = "1O86_native_lig.mol2"

for f in [RECEPTOR_PDB, RECEPTOR_PDBQT, NATIVE_LIG_MOL2]:
    exists = os.path.exists(f)
    print(f, "exists:", exists)
    if not exists:
        print(f"  -> {f} not found in {DOCKING_DIR}. If this project's files")
        print("     live somewhere else, update DOCKING_DIR above to match,")
        print("     or run `find / -name \'" + f + "\' 2>/dev/null` in a terminal")
        print("     to locate it.")


Working directory: /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results
1O86_clean_H.pdb exists: True
1O86_clean_H.pdbqt exists: True
1O86_native_lig.mol2 exists: True


## Shared RMSD utility (MCS-based, proven robust for lisinopril throughout this project)

`GetBestRMS`/bond-order-based comparison has repeatedly failed on this
specific ligand due to mol2 charge/bond-order guessing problems. This
element-only, bond-order-agnostic method is used for consistency with
every other lisinopril RMSD calculation in this study.

In [2]:
def rmsd_via_mcs(ref, probe):
    mcs = rdFMCS.FindMCS(
        [ref, probe],
        atomCompare=rdFMCS.AtomCompare.CompareElements,
        bondCompare=rdFMCS.BondCompare.CompareAny,
        ringMatchesRingOnly=False,
        matchValences=False,
        timeout=10,
    )
    patt = Chem.MolFromSmarts(mcs.smartsString)
    ref_match = ref.GetSubstructMatch(patt)
    probe_match = probe.GetSubstructMatch(patt)
    if not ref_match or not probe_match:
        return None, 0
    ref_conf = ref.GetConformer()
    probe_conf = probe.GetConformer()
    diffs = []
    for ai, bi in zip(ref_match, probe_match):
        p1 = np.array(ref_conf.GetAtomPosition(ai))
        p2 = np.array(probe_conf.GetAtomPosition(bi))
        diffs.append(np.sum((p1 - p2) ** 2))
    rmsd = round(float(np.sqrt(np.mean(diffs))), 3)
    return rmsd, len(ref_match)

ref_mol_raw = Chem.MolFromMol2File(NATIVE_LIG_MOL2, sanitize=False, removeHs=True)
print("Reference ligand loaded, atom count:", ref_mol_raw.GetNumAtoms())


Reference ligand loaded, atom count: 29


[16:20:21] 1O86: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.


---
## 1. Smina redocking

Using the same box already established and reused throughout every ACE
redocking attempt in this project (center 40.553, 32.798, 47.285; size
15.632, 20.048, 21.865), and the same exhaustiveness (10) used for
this target's own compound-screening protocol, for direct comparability
with the existing Vina results.

In [3]:
SMINA_BIN = BIN + "smina"
assert os.path.exists(SMINA_BIN), f"{SMINA_BIN} does not exist -- check the bin folder."

smina_out = "1O86_native_lig_smina_out.sdf"

r = subprocess.run([
    SMINA_BIN,
    "-r", RECEPTOR_PDB,
    "-l", NATIVE_LIG_MOL2,
    "-o", smina_out,
    "--center_x", "40.553", "--center_y", "32.798", "--center_z", "47.285",
    "--size_x", "15.632", "--size_y", "20.048", "--size_z", "21.865",
    "--exhaustiveness", "10",
    "--num_modes", "10",
], capture_output=True, text=True)

print("stdout:", r.stdout)
print("stderr:", r.stderr)
print("return code:", r.returncode)


stdout:    _______  _______ _________ _        _______ 
  (  ____ \(       )\__   __/( (    /|(  ___  )
  | (    \/| () () |   ) (   |  \  ( || (   ) |
  | (_____ | || || |   | |   |   \ | || (___) |
  (_____  )| |(_)| |   | |   | (\ \) ||  ___  |
        ) || |   | |   | |   | | \   || (   ) |
  /\____) || )   ( |___) (___| )  \  || )   ( |
  \_______)|/     \|\_______/|/    )_)|/     \|


smina is based off AutoDock Vina. Please cite appropriately.

Weights      Terms
-0.035579    gauss(o=0,_w=0.5,_c=8)
-0.005156    gauss(o=3,_w=2,_c=8)
0.840245     repulsion(o=0,_c=8)
-0.035069    hydrophobic(g=0.5,_b=1.5,_c=8)
-0.587439    non_dir_h_bond(g=-0.7,_b=0,_c=8)
1.923        num_tors_div

Using random seed: 1762673678

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+---

In [4]:
# Smina writes minimizedAffinity as an SDF property per pose -- read all
# poses, extract scores, and compute RMSD for each against the crystal
# reference, same as every other redocking table in this project.
smina_poses = Chem.SDMolSupplier(smina_out, removeHs=True, sanitize=False)

smina_rows = []
for i, pose in enumerate(smina_poses):
    if pose is None:
        smina_rows.append({"Pose": i + 1, "Score_kcal_mol": None, "RMSD_A": None})
        continue
    score = None
    if pose.HasProp("minimizedAffinity"):
        score = float(pose.GetProp("minimizedAffinity"))
    rmsd, n_matched = rmsd_via_mcs(ref_mol_raw, pose)
    print(f"Pose {i+1}: score={score}, matched {n_matched}/{ref_mol_raw.GetNumAtoms()} atoms, RMSD={rmsd}")
    smina_rows.append({"Pose": i + 1, "Score_kcal_mol": score, "RMSD_A": rmsd})

smina_df = pd.DataFrame(smina_rows).sort_values("Score_kcal_mol").reset_index(drop=True)
print()
print("=== Smina results, ranked by score ===")
display(smina_df)

top_smina = smina_df.iloc[0]
print(f"\nTop-scoring pose: score={top_smina['Score_kcal_mol']}, RMSD={top_smina['RMSD_A']} A")
print("PASS" if (top_smina['RMSD_A'] is not None and top_smina['RMSD_A'] < 2.0) else "FAIL")


Pose 1: score=-8.72401, matched 29/29 atoms, RMSD=1.314
Pose 2: score=-8.64951, matched 29/29 atoms, RMSD=7.041
Pose 3: score=-8.64287, matched 29/29 atoms, RMSD=7.836
Pose 4: score=-8.59178, matched 29/29 atoms, RMSD=9.522
Pose 5: score=-8.55475, matched 29/29 atoms, RMSD=7.216
Pose 6: score=-8.51987, matched 29/29 atoms, RMSD=8.408
Pose 7: score=-8.51116, matched 29/29 atoms, RMSD=8.26
Pose 8: score=-8.49618, matched 29/29 atoms, RMSD=8.49
Pose 9: score=-8.46607, matched 29/29 atoms, RMSD=8.309
Pose 10: score=-8.42158, matched 29/29 atoms, RMSD=8.155

=== Smina results, ranked by score ===


[16:20:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:20:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:20:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:20:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:20:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:20:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:20:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:20:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:20:37] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

,Pose,Score_kcal_mol,RMSD_A
0,1,-8.72401,1.314
1,2,-8.64951,7.041
2,3,-8.64287,7.836
3,4,-8.59178,9.522
4,5,-8.55475,7.216
5,6,-8.51987,8.408
6,7,-8.51116,8.260
7,8,-8.49618,8.490
8,9,-8.46607,8.309
9,10,-8.42158,8.155



Top-scoring pose: score=-8.72401, RMSD=1.314 A
PASS


---
## 2. LeDock redocking

LeDock requires a `dock.in` configuration file (built via
`generate_ledock_file`) rather than command-line flags, and produces a
`.dok` results file that must be converted to SDF before analysis --
following the origin template's pipeline exactly (Section 6).

In [5]:
cmd.load(filename=RECEPTOR_PDB, format='pdb', object='prot')
cmd.load(filename=NATIVE_LIG_MOL2, format='mol2', object='lig')

X, Y, Z = getbox(selection='lig', extending=5.0, software='ledock')
cmd.delete('all')
print(X, '\n', Y, '\n', Z)


{'minX': 32.73699951171875, 'maxX': 48.36899948120117} 
 {'minY': 22.77400016784668, 'maxY': 42.821998596191406} 
 {'minZ': 36.35300064086914, 'maxZ': 58.21799850463867}


In [6]:
generate_ledock_file(
    receptor=RECEPTOR_PDB,
    x=[X['minX'], X['maxX']],
    y=[Y['minY'], Y['maxY']],
    z=[Z['minZ'], Z['maxZ']],
    n_poses=10,
    rmsd=1.0,
    l_list=[NATIVE_LIG_MOL2],
    l_list_outfile='ledock_ligand_list.txt',
    out='dock.in',
)

LEDOCK_BIN = BIN + "ledock_linux_x86"
assert os.path.exists(LEDOCK_BIN), f"{LEDOCK_BIN} does not exist -- check the bin folder."

r = subprocess.run([LEDOCK_BIN, "dock.in"], capture_output=True, text=True)
print("stdout:", r.stdout)
print("stderr:", r.stderr)
print("return code:", r.returncode)


stdout: 
stderr: 
return code: 0


In [1]:
dok_file = NATIVE_LIG_MOL2.replace('.mol2', '.dok')
assert os.path.exists(dok_file), (
    f"{dok_file} not found -- check the LeDock run output above for the "
    "actual .dok filename it produced."
)

ledock_out_sdf = "1O86_native_lig_ledock_out.sdf"
dok_to_sdf(dok_file=dok_file, output=ledock_out_sdf)

ledock_poses = Chem.SDMolSupplier(ledock_out_sdf, removeHs=True, sanitize=False)

ledock_rows = []
for i, pose in enumerate(ledock_poses):
    if pose is None:
        ledock_rows.append({"Pose": i + 1, "Score_kcal_mol": None, "RMSD_A": None})
        continue
    score = None
    for prop_name in ["Score", "score"]:
        if pose.HasProp(prop_name):
            score = float(pose.GetProp(prop_name))
            break
    rmsd, n_matched = rmsd_via_mcs(ref_mol_raw, pose)
    print(f"Pose {i+1}: score={score}, matched {n_matched}/{ref_mol_raw.GetNumAtoms()} atoms, RMSD={rmsd}")
    ledock_rows.append({"Pose": i + 1, "Score_kcal_mol": score, "RMSD_A": rmsd})

ledock_df = pd.DataFrame(ledock_rows)
if ledock_df["Score_kcal_mol"].notna().any():
    ledock_df = ledock_df.sort_values("Score_kcal_mol").reset_index(drop=True)
print()
print("=== LeDock results ===")
display(ledock_df)

top_ledock = ledock_df.iloc[0]
print(f"\nTop-ranked pose: score={top_ledock['Score_kcal_mol']}, RMSD={top_ledock['RMSD_A']} A")
print("PASS" if (top_ledock['RMSD_A'] is not None and top_ledock['RMSD_A'] < 2.0) else "FAIL")


NameError: name 'NATIVE_LIG_MOL2' is not defined

---
## 3. Cross-engine comparison

Only run this cell after both engines above have completed successfully
-- it compares real results, not placeholders.

In [ ]:
print("=== Cross-engine redocking comparison: ACE (1O86) / lisinopril ===")
print(f"{'Engine':<12} {'Top score':<12} {'Top-pose RMSD (A)':<20} {'Result'}")
print(f"{'Vina':<12} {'-7.876 to -7.918':<12} {'7.8-9.5':<20} {'FAIL (established)'}")

smina_top = smina_df.iloc[0]
smina_result = 'PASS' if smina_top['RMSD_A'] is not None and smina_top['RMSD_A'] < 2.0 else 'FAIL'
print(f"{'Smina':<12} {str(smina_top['Score_kcal_mol']):<12} {str(smina_top['RMSD_A']):<20} {smina_result}")

ledock_top = ledock_df.iloc[0]
ledock_result = 'PASS' if ledock_top['RMSD_A'] is not None and ledock_top['RMSD_A'] < 2.0 else 'FAIL'
print(f"{'LeDock':<12} {str(ledock_top['Score_kcal_mol']):<12} {str(ledock_top['RMSD_A']):<20} {ledock_result}")

# Save for the manuscript
comparison_df = pd.DataFrame([
    {"Engine": "Vina", "Top_score": "-7.876 to -7.918", "Top_pose_RMSD_A": "7.8-9.5", "Result": "FAIL"},
    {"Engine": "Smina", "Top_score": smina_top['Score_kcal_mol'], "Top_pose_RMSD_A": smina_top['RMSD_A'], "Result": smina_result},
    {"Engine": "LeDock", "Top_score": ledock_top['Score_kcal_mol'], "Top_pose_RMSD_A": ledock_top['RMSD_A'], "Result": ledock_result},
])
comparison_df.to_csv("ACE_1O86_cross_engine_redocking_comparison.csv", index=False)
print()
print("Saved: ACE_1O86_cross_engine_redocking_comparison.csv")
